<a href="https://colab.research.google.com/github/LinearAlgebruh/PredictingMarchMadness/blob/main/ProjectDataPrep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import torch
!pip install pyreadr
!pip install rapidfuzz
import pyreadr
from rapidfuzz import process, fuzz



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.3/788.3 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.3 MB/s eta 0:00:00


In [2]:
def readRSD(url, scratch):
  local = pyreadr.download_file(url, scratch)
  result = pyreadr.read_r(local)
  df = result[None]
  return df

RegSzn2014 = readRSD("https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/RegSeasonStats/MMStats_2014.rds", "./2014SeasonStats.rda")
AdvSt2014 = readRSD("https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/RatingStats/RatingStats_2014.rds", "./2014RatingStats.rda" )

# ids = readRSD("https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/Teams.rds", "./teamIds.rda")
# df_ids = ids[['TeamID', 'TeamName']]
# df_ids.columns = ['id', 'School']
# df_ids.head()

df_ids = pd.read_csv('https://github.com/LinearAlgebruh/PredictingMarchMadness/raw/refs/heads/main/NCAATeamIds.csv')



In [36]:
# Join without exact matches. Still struggles with abbreviations
def fuzzy_merge(df_left, df_right, key_left, key_right, threshold=60, limit=1):
    """
    Merge data frames on key without exact matches. Used AI for this
    """
    right_names = df_right[key_right].tolist()

    def get_best_match(name):
        match = process.extractOne(
            name, right_names,
            scorer=fuzz.token_sort_ratio,   # handles word-order differences
            score_cutoff=threshold
        )
        return match[0] if match else None

    df_left = df_left.copy()
    df_left['_match_key'] = df_left[key_left].apply(get_best_match)

    merged = df_left.merge(
        df_right,
        left_on='_match_key',
        right_on=key_right,
        how='left'
    ).drop(columns=['_match_key'])

    return merged

dfComplete = fuzzy_merge(RegSzn2014, df_ids, 'School','School')
# dfComplete = fuzzy_merge(dfComplete)
dfComplete.iloc[[167]]

df_with_nans = dfComplete[dfComplete['TeamID'].isna()]
df_with_nans


,Rk,School_x,G,W,L,W.L.,SRS,SOS,W.1,L.1,...,AST,STL,BLK,TOV,PF,TeamID,TeamName,School_y,FirstD1Season,LastD1Season
158,159,Loyola (IL),32,10,22,0.313,-4.75,-1.14,4,14,...,419,176,82,426,622,NaN,NaN,NaN,NaN,NaN
160,161,Loyola (MD),30,11,19,0.367,-10.58,-3.23,6,12,...,336,236,128,357,584,NaN,NaN,NaN,NaN,NaN
166,167,Maryland-Baltimore County,30,9,21,0.300,-14.21,-6.55,5,11,...,355,240,84,450,702,NaN,NaN,NaN,NaN,NaN
175,176,Miami (OH),31,13,18,0.419,-4.37,0.13,8,10,...,387,267,73,420,537,NaN,NaN,NaN,NaN,NaN
205,206,NJIT,29,13,16,0.448,-12.52,-7.48,0,0,...,394,187,104,379,585,NaN,NaN,NaN,NaN,NaN
238,239,Pennsylvania,28,8,20,0.286,-6.59,-0.01,5,9,...,408,163,97,450,633,NaN,NaN,NaN,NaN,NaN
328,329,VMI,35,22,13,0.629,-5.53,-6.33,11,5,...,491,254,218,371,661,NaN,NaN,NaN,NaN,NaN


In [23]:
url = "https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/RegSeasonStats/MMStats_2014.rds"
scratch = "./2014SeasonStats.rda"
local = pyreadr.download_file(url, scratch)
result = pyreadr.read_r(local)
df = result[None]
type(df["School"][0])

print(df['School'].dtype)




# df.join(df2, on="School")

# df.head()


object


In [3]:
url = "https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/RatingStats/RatingStats_2014.rds"
scratch = "./2014RatingStats.rda"
local = pyreadr.download_file(url, scratch)
result = pyreadr.read_r(local)
df2 = result[None]

type(df2['School'][0])
print(df2['School'].dtype)

dfFull = pd.merge(df, df2, on = "School")

dfFull.shape



object


NameError: name 'df' is not defined

In [8]:
df = readRSD('https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/Prelim2019_RegularSeasonDetailedResults.rds', './DetailedStats.rda')

# Scrape all box scores for a given team from a single season. I think this is the
# best way to get season level data for us.
def get_team_stats(df, team_id, year=None):
    """
    Returns all stats for a given team from a Detailed Results DataFrame,
    normalized to team/opponent perspective (instead of winner/loser).

    Parameters:
        df      : Detailed Results DataFrame
        team_id : The team's ID to filter by
        year    : (optional) Season year to filter by. If None, returns all years.

    Returns:
        DataFrame with one row per game, stats from the team's perspective
    """

    stat_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

    # --- Games where team won ---
    won = df[(df['WTeamID'] == team_id) & (df['Season'] == year)].copy()
    won['TeamID']      = won['WTeamID']
    won['OpponentID']  = won['LTeamID']
    won['TeamScore']   = won['WScore']
    won['OppScore']    = won['LScore']
    won['Result']      = 'W'
    # won['Location']    = won['WLoc']
    for col in stat_cols:
        won[col]           = won[f'W{col}']
        won[f'Opp{col}']   = won[f'L{col}']

    # --- Games where team lost ---
    lost = df[(df['LTeamID'] == team_id) & (df['Season'] == year)].copy()
    lost['TeamID']     = lost['LTeamID']
    lost['OpponentID'] = lost['WTeamID']
    lost['TeamScore']  = lost['LScore']
    lost['OppScore']   = lost['WScore']
    lost['Result']     = 'L'
    # Flip location: if winner was Home, this team was Away, and vice versa
    # lost['Location']   = lost['WLoc'].map({'H': 'A', 'A': 'H', 'N': 'N'})
    for col in stat_cols:
        lost[col]          = lost[f'L{col}']
        lost[f'Opp{col}']  = lost[f'W{col}']

    # --- Combine and select final columns ---
    keep_cols = ['Season', 'DayNum', 'TeamID', 'OpponentID', 'TeamScore',
                 'OppScore', 'Result', 'NumOT'] + \
                stat_cols + [f'Opp{col}' for col in stat_cols]

    team_games = pd.concat([won, lost])[keep_cols]


    return team_games.sort_values(['Season', 'DayNum']).reset_index(drop=True)


df1104 = get_team_stats(df, 1104, 2014)
df1104.head()


,Season,DayNum,TeamID,OpponentID,TeamScore,OppScore,Result,NumOT,FGM,FGA,...,OppFGA3,OppFTM,OppFTA,OppOR,OppDR,OppAst,OppTO,OppStl,OppBlk,OppPF
0,2014,4,1104,1328,73,82,L,0,29,61,...,21,19,25,17,21,16,9,7,5,18
1,2014,10,1104,1403,76,64,W,0,28,51,...,20,4,8,13,13,10,16,8,5,15
2,2014,15,1104,1209,75,58,W,0,25,52,...,17,9,14,7,17,11,13,8,4,21
3,2014,23,1104,1181,64,74,L,0,23,57,...,15,24,29,8,27,15,19,10,1,20
4,2014,25,1104,1180,83,85,L,3,27,64,...,17,25,31,18,25,15,14,6,3,29
